# Applied Linear Algebra Assignment

Pure Python implementation — **no NumPy, Pandas, or external math libraries**.
All vectors, matrices, and operations (determinants, matrix-vector multiplication,
matrix-matrix multiplication) are implemented from scratch using native Python
lists and loops.

- **Task 1: The Restricted Robot** — Span & Basis
- **Task 2: The Mystery Black Box** — Linear Transformations


In [1]:
from fractions import Fraction  # stdlib only — lets us test "is this a whole number"
                                 # exactly, with no floating point error.

## Task 1: The Restricted Robot (Span & Basis)

The robot can only move using **whole-number multiples** of two vectors:

- **Move A** = (2, 1) &nbsp;→ 2 steps Forward, 1 step Right
- **Move B** = (1, −2) &nbsp;→ 1 step Forward, 2 steps Left (Left = −Right)

A target coordinate `(x, y)` is reachable if there exist **integers** `a, b` such that:

$$a \cdot A + b \cdot B = (x, y)$$

Written as a linear system:

$$
\begin{bmatrix} A_x & B_x \\ A_y & B_y \end{bmatrix}
\begin{bmatrix} a \\ b \end{bmatrix}
=
\begin{bmatrix} x \\ y \end{bmatrix}
$$

We solve this 2×2 system by hand using **Cramer's Rule** (determinants), which
avoids needing any matrix-inversion library.

In [2]:
MOVE_A = (2, 1)     # 2 Forward, 1 Right
MOVE_B = (1, -2)    # 1 Forward, 2 Left


def determinant_2x2(matrix):
    """matrix = [[a, b], [c, d]]  ->  a*d - b*c"""
    (a, b), (c, d) = matrix
    return a * d - b * c

In [3]:
def solve_2x2_system(move_a, move_b, target):
    """
    Solve  a*move_a + b*move_b = target  for (a, b) using Cramer's Rule.
    Returns (a, b) as Fractions (exact, no floating point drift), or None
    if the two moves do not span the plane (determinant == 0).
    """
    ax, ay = move_a
    bx, by = move_b
    tx, ty = target

    # Coefficient matrix M = [[ax, bx], [ay, by]]
    M = [[ax, bx], [ay, by]]
    det_M = determinant_2x2(M)

    if det_M == 0:
        return None  # Move A and Move B are parallel -> they don't span R^2

    # Cramer's rule: replace each column of M with the target vector in turn.
    M_a = [[tx, bx], [ty, by]]   # replace column for 'a' with target
    M_b = [[ax, tx], [ay, ty]]   # replace column for 'b' with target

    a = Fraction(determinant_2x2(M_a), det_M)
    b = Fraction(determinant_2x2(M_b), det_M)
    return a, b

In [4]:
def is_reachable(target, move_a=MOVE_A, move_b=MOVE_B, verbose=True):
    """
    Determine whether `target` (x, y) is reachable using INTEGER multiples
    of move_a and move_b.
    """
    solution = solve_2x2_system(move_a, move_b, target)

    if solution is None:
        if verbose:
            print(f"Target {target}: UNREACHABLE (Move A and Move B do not "
                  f"span the 2D grid - they are parallel/dependent).")
        return False

    a, b = solution
    reachable = (a.denominator == 1) and (b.denominator == 1)

    if verbose:
        status = "REACHABLE" if reachable else "UNREACHABLE"
        print(f"Target {target}: a = {a}, b = {b}  ->  {status}"
              + ("" if reachable else "  (needs non-integer move counts)"))

    return reachable

In [5]:
def basis_check(move_a=MOVE_A, move_b=MOVE_B):
    """
    Move A and Move B form a BASIS of R^2 (the real-number plane) if and
    only if they are linearly independent, i.e. determinant != 0.
    Note: forming a basis of R^2 is NOT the same as being able to reach
    every INTEGER coordinate - that requires the integer lattice condition
    checked in is_reachable().
    """
    det = determinant_2x2([[move_a[0], move_b[0]], [move_a[1], move_b[1]]])
    print(f"det([A | B]) = {det}")
    if det != 0:
        print("Move A and Move B ARE linearly independent -> they form a "
              "basis for the full 2D plane (R^2).")
        print("However, since only INTEGER combinations are allowed, the "
              "robot can only reach a sub-lattice of integer points, not "
              "every integer coordinate (see is_reachable tests below).")
    else:
        print("Move A and Move B are linearly DEPENDENT -> they only span "
              "a line, not the full plane.")
    return det

### Run the basis check

In [6]:
print(f"Move A = {MOVE_A}, Move B = {MOVE_B}\n")
basis_check()

Move A = (2, 1), Move B = (1, -2)

det([A | B]) = -5
Move A and Move B ARE linearly independent -> they form a basis for the full 2D plane (R^2).
However, since only INTEGER combinations are allowed, the robot can only reach a sub-lattice of integer points, not every integer coordinate (see is_reachable tests below).


-5

### Test reachability for several target coordinates

In [7]:
test_targets = [
    (3, -1),   # 1*A + 1*B -> should be reachable
    (5, 0),    # 2*A + 1*B -> reachable
    (1, 1),    # check
    (0, 0),    # trivially reachable (a=b=0)
    (4, 3),    # likely unreachable
    (100, 50),
]

for t in test_targets:
    is_reachable(t)

Target (3, -1): a = 1, b = 1  ->  REACHABLE
Target (5, 0): a = 2, b = 1  ->  REACHABLE
Target (1, 1): a = 3/5, b = -1/5  ->  UNREACHABLE  (needs non-integer move counts)
Target (0, 0): a = 0, b = 0  ->  REACHABLE
Target (4, 3): a = 11/5, b = -2/5  ->  UNREACHABLE  (needs non-integer move counts)
Target (100, 50): a = 50, b = 0  ->  REACHABLE


**Try your own target!** Change the coordinates below and re-run.

In [8]:
my_target = (7, -4)   # <-- edit this
is_reachable(my_target)

Target (7, -4): a = 2, b = 3  ->  REACHABLE


True

## Task 2: The Mystery Black Box (Linear Transformations)

We're told:

- T(1, 0) = (0, 1)
- T(0, 1) = (−1, 0)

For any linear transformation, the transformation matrix `M` is built by
placing the **images of the standard basis vectors as its columns**:

$$
M = \begin{bmatrix} T(1,0) & T(0,1) \end{bmatrix}
= \begin{bmatrix} 0 & -1 \\ 1 & 0 \end{bmatrix}
$$

This is exactly a **90° counter-clockwise rotation matrix**.

In [9]:
def matvec_mult(matrix, vector):
    """Raw matrix-vector multiplication: M (2x2) * v (2x1) -> vector (2,)"""
    result = []
    for row in matrix:
        total = 0
        for coeff, component in zip(row, vector):
            total += coeff * component
        result.append(total)
    return tuple(result)


def matmat_mult(m1, m2):
    """
    Raw 2x2 matrix-matrix multiplication.
    Composing transformations: applying m2 first, then m1, is the same as
    the single transformation (m1 @ m2).
    """
    rows_m1, cols_m1 = len(m1), len(m1[0])
    rows_m2, cols_m2 = len(m2), len(m2[0])
    assert cols_m1 == rows_m2, "Incompatible matrix dimensions"

    result = [[0] * cols_m2 for _ in range(rows_m1)]
    for i in range(rows_m1):
        for j in range(cols_m2):
            total = 0
            for k in range(cols_m1):
                total += m1[i][k] * m2[k][j]
            result[i][j] = total
    return result


def deduce_matrix(t_e1, t_e2):
    """
    Deduce the 2x2 transformation matrix from the images of the standard
    basis vectors e1=(1,0) and e2=(0,1).
    """
    return [
        [t_e1[0], t_e2[0]],
        [t_e1[1], t_e2[1]],
    ]


def print_matrix(matrix, label="Matrix"):
    print(f"{label}:")
    for row in matrix:
        print("  [", "  ".join(f"{v:>3}" for v in row), "]")

### Deduce the black box's matrix

In [10]:
BLACK_BOX_MATRIX = deduce_matrix((0, 1), (-1, 0))
print_matrix(BLACK_BOX_MATRIX, "Deduced transformation matrix M")
print("(This is a 90-degree counter-clockwise rotation matrix.)")

Deduced transformation matrix M:
  [   0   -1 ]
  [   1    0 ]
(This is a 90-degree counter-clockwise rotation matrix.)


### Apply it to a test vector

In [11]:
test_vector = (2, 1)
print(f"Applying M once to {test_vector}:")
print(" ", matvec_mult(BLACK_BOX_MATRIX, test_vector))

Applying M once to (2, 1):
  (-1, 2)


### Observe the compounding effect of repeated application
Applying a 90° rotation four times should return the vector to its start (360°).

In [12]:
def apply_black_box_repeatedly(vector, times, matrix=BLACK_BOX_MATRIX):
    """
    Apply the black box transformation `times` times in sequence,
    printing the state after each application (observing compounding
    effects, e.g. rotation accumulation).
    """
    current = vector
    print(f"Start: {current}")
    for step in range(1, times + 1):
        current = matvec_mult(matrix, current)
        print(f"After application {step}: {current}")
    return current

apply_black_box_repeatedly(test_vector, 4)

Start: (2, 1)
After application 1: (-1, 2)
After application 2: (-2, -1)
After application 3: (1, -2)
After application 4: (2, 1)


(2, 1)

### Compose the transformation with itself via matrix multiplication
`M @ M` should equal a 180° rotation, and applying it once should match applying `M` twice.

In [13]:
M_squared = matmat_mult(BLACK_BOX_MATRIX, BLACK_BOX_MATRIX)
print_matrix(M_squared, "M @ M")

print(f"\nCheck: M_squared applied to {test_vector} ->",
      matvec_mult(M_squared, test_vector))
print(f"Direct double-application result should match: ",
      matvec_mult(BLACK_BOX_MATRIX, matvec_mult(BLACK_BOX_MATRIX, test_vector)))

M @ M:
  [  -1    0 ]
  [   0   -1 ]

Check: M_squared applied to (2, 1) -> (-2, -1)
Direct double-application result should match:  (-2, -1)


**Try your own vector!** Change it below and re-run to see the black box in action.

In [14]:
my_vector = (3, -2)   # <-- edit this
matvec_mult(BLACK_BOX_MATRIX, my_vector)

(2, 3)